# K1: Planification Urbaine

## 1. Description du problème

Ce sujet porte sur la **planification urbaine optimisée**. L'objectif est de déterminer les emplacements idéaux pour diverses infrastructures (hôpitaux, écoles, centres commerciaux, parcs, stations de recharge) au sein d'une zone urbaine.

Il s'agit d'un problème classique de **Théorie de la Localisation** (p-median, p-center, MCLP), où l'on cherche à maximiser la couverture de la population tout en respectant des contraintes de budget, de superficie, de couverture équitable, de distance maximum et de compatibilité entre sites.

Ce notebook présente une solution d'optimisation pour le placement d'infrastructures sur des données urbaines réelles (à Lyon) en utilisant la Programmation par Contraintes (CP-SAT).

Les principales resources utilisées :
- **Optimisation** : `ortools` (module `cp_model`). Nécessaire pour ne pas avoir a reimplementer l'algorithme CP-SAT.
- **Géographie & Cartographie** : `geopandas` (manipulation de données spatiales), `folium` (cartes interactives), `osmnx` (extraction de données OpenStreetMap). Utiles pour charger les données de la ville de Lyon et tester le solveur. On utilise **OpenStreetMap (OSM)** pour le réseau routier et les infrastructures existantes et **INSEE** pour les données de population par quartier/carreau.

In [2]:
import folium
import geopandas as gpd
import os
import osmnx as ox
from ortools.sat.python import cp_model
import math
from shapely.geometry import box

## 2. Algorithme : CP-SAT

### Description du solveur OR-Tools CP-SAT

OR-Tools CP-SAT (Constraint Programming SAT) est un solveur de contraintes très performant développé par Google. Il est utilisé pour résoudre efficacement des problèmes complexes en optimisation et satisfaction de contraintes. On le retrouve dans des applications d'ordonnancement, d'affectation de ressources et d'autres problèmes combinatoires.

ORTools permet de modéliser des problèmes de décision sur des domaines finis, à l'aide de variables entières, de variables booléennes et de clauses. La librairie dispose d'un large panel de contraintes, dont des contraintes globales (alldifferent, sum, circuit, cumulative, regular, table, reservoir, etc.).

OR-Tools CP-SAT combine différentes approches : Programmation Par Contraintes (CP), SAT, mais aussi de la Programmation Linéraire. L'hybridation CP-SAT est basée sur le framework de Génération Paresseuse de Clauses (LCG), mis en avant par le solveur Chuffed, qui aide le solveur a analyser ses propres conflits durant la recherche d'une solution afin de mieux apprendre de ses erreurs. La Programmation Linéaire quant à elle aide le solveur à mieux exploiter la fonction d'optimisation.

Le solveur utilise par défaut un Portfolio de solveurs permettant de combiner simultanément différentes configurations afin de résoudre le problème le plus efficacement possible.

### Modélisation CP-SAT

TODO: Variables binaires, Contraintes d'assignation, Contraintes de capacité, Objectif, ...

## 3. Chargement des données de Lyon

TODO: Pourquoi Lyon

In [ ]:
def fetch_data(
    city: str,
    output_dir: str = "data",
    district_count: int = 12,
    district_mode: str = "predefined",
) -> None:
    """Fetches data for the specified city.

    Fetches the road network data for the specified city, and saves it as a GeoJSON file.
    creates three files : 
    - roads.geojson : the road network of the city
    - districts.geojson : the districts of the city
    - sites.geojson : sites we want to cover (schools, hospitals, etc.)
    
    Parameters
    ----------
    city : str
        The city for which to fetch data.
    output_dir : str, default="data"
        The directory where the fetched data will be saved.
    district_count : int, default=12
        The approximate number of districts to generate.
    district_mode : str, default="predefined"
        Either "predefined" to use existing administrative districts or "grid" to force a rectangular split.
    
    """

    print(f"Fetching data for {city} in {output_dir}...")
    os.makedirs(output_dir, exist_ok=True)

    city_boundary = ox.geocode_to_gdf(city)[["geometry"]].copy()
    if city_boundary.crs is not None and city_boundary.crs.to_string() != "EPSG:4326":
        city_boundary = city_boundary.to_crs(epsg=4326)
    city_boundary_geom = city_boundary.geometry.iloc[0]
    districts_path = os.path.join(output_dir, "districts.geojson")

    graph = ox.graph_from_place(city, network_type="drive")
    roads = ox.graph_to_gdfs(graph, nodes=False)[["geometry"]].copy()
    if roads.crs is not None and roads.crs.to_string() != "EPSG:4326":
        roads = roads.to_crs(epsg=4326)
    roads_path = os.path.join(output_dir, "roads.geojson")
    roads.to_file(roads_path, driver="GeoJSON")
    city_roads_path = os.path.join(output_dir, f"{city.replace(' ', '_')}_roads.geojson")
    roads.to_file(city_roads_path, driver="GeoJSON")

    road_union = roads.geometry.union_all()
    districts = gpd.GeoDataFrame()

    if district_mode == "predefined":
        district_tags = {
            "boundary": "administrative",
            "admin_level": ["10", "9", "8"],
        }
        candidate_districts = ox.features_from_place(city, district_tags)
        if not candidate_districts.empty:
            candidate_districts = candidate_districts[candidate_districts.geometry.notna()].copy()
            if candidate_districts.crs is not None and candidate_districts.crs.to_string() != "EPSG:4326":
                candidate_districts = candidate_districts.to_crs(epsg=4326)
            candidate_districts = candidate_districts[candidate_districts.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()
            candidate_districts = candidate_districts.explode(index_parts=False, ignore_index=True)
            candidate_districts = candidate_districts[candidate_districts.geometry.intersects(city_boundary_geom)].copy()
            candidate_districts["geometry"] = candidate_districts.geometry.intersection(city_boundary_geom)
            candidate_districts = candidate_districts[~candidate_districts.geometry.is_empty].copy()
            candidate_districts = candidate_districts[candidate_districts.geometry.intersects(road_union)].copy()
            candidate_districts = candidate_districts.drop_duplicates(subset=["geometry"]).reset_index(drop=True)
            if not candidate_districts.empty:
                districts = candidate_districts[["geometry"]].copy()
                districts = gpd.GeoDataFrame(districts, geometry="geometry", crs="EPSG:4326")

    if districts.empty:
        rows = max(1, int(math.floor(math.sqrt(district_count))))
        cols = max(1, int(math.ceil(district_count / rows)))
        minx, miny, maxx, maxy = city_boundary_geom.bounds
        cell_width = (maxx - minx) / cols if cols else (maxx - minx)
        cell_height = (maxy - miny) / rows if rows else (maxy - miny)
        district_geometries = []
        for row in range(rows):
            for col in range(cols):
                cell = box(
                    minx + col * cell_width,
                    miny + row * cell_height,
                    minx + (col + 1) * cell_width,
                    miny + (row + 1) * cell_height,
                )
                district = cell.intersection(city_boundary_geom)
                if district.is_empty:
                    continue
                if not district.intersects(road_union):
                    continue
                district_geometries.append(district)

        if not district_geometries:
            district_geometries = [city_boundary_geom]

        districts = gpd.GeoDataFrame(geometry=district_geometries, crs="EPSG:4326")
    districts.to_file(districts_path, driver="GeoJSON")

    site_tags = {
        "amenity": ["school", "hospital", "clinic", "university", "kindergarten", "library", "college"],
        "leisure": ["park", "sports_centre"],
        "shop": ["supermarket"],
    }
    sites = ox.features_from_place(city, site_tags)
    if not sites.empty:
        sites = sites[sites.geometry.notna()].copy()
        if sites.crs is not None and sites.crs.to_string() != "EPSG:4326":
            sites = sites.to_crs(epsg=4326)
        sites["geometry"] = sites.geometry.representative_point()
        site_columns = [column for column in ["geometry", "name", "amenity", "leisure", "shop"] if column in sites.columns]
        sites = sites[site_columns].copy()
    else:
        sites = districts[["geometry"]].copy()
        sites["name"] = city
        sites["amenity"] = "city_boundary"

    district_union = districts.geometry.union_all()
    sites = sites[sites.geometry.apply(district_union.covers)].copy()

    sites_path = os.path.join(output_dir, "sites.geojson")
    sites.to_file(sites_path, driver="GeoJSON")

    print(
        f"Data for {city} saved to {districts_path}, {roads_path} and {sites_path}."
    )

In [4]:
def visualize_data(city: str, data_dir: str = "data") -> None:
    """Visualizes the data for the specified city.

    Visualizes the road network data for the specified city using Folium.
    Show the result in the console.

    Parameters
    ----------
    city : str
        The city for which to visualize data.
    data_dir : str, default="data"
        The directory where the fetched data is saved.
    """
    print(f"Visualizing data for {city} from {data_dir}...")
    data_path = os.path.join(data_dir, f"{city.replace(' ', '_')}_roads.geojson")
    gdf = gpd.read_file(data_path)

    if gdf.crs is not None and gdf.crs.to_string() != "EPSG:4326":
        gdf = gdf.to_crs(epsg=4326)

    road_geometry = gdf[["geometry"]].copy()
    bounds = road_geometry.total_bounds
    center_lat = (bounds[1] + bounds[3]) / 2
    center_lon = (bounds[0] + bounds[2]) / 2

    m = folium.Map(location=[center_lat, center_lon], zoom_start=12)
    folium.GeoJson(road_geometry, name="road network").add_to(m)
    display(m)

In [ ]:
CITY = "Lyon, France"
DISTRICT_COUNT = 12
fetch_data(city=CITY, district_count=DISTRICT_COUNT)

# Beware, this map will not be displayed in VSCODE,
# you need the geo-data-viewer extension and look at the json file

# visualize_data(city=CITY)

Fetching data for Lyon, France in data...
Data for Lyon, France saved to data\districts.geojson, data\roads.geojson and data\sites.geojson.


## 4. Solveur CP-SAT

TODO: Explication rapide du code

In [6]:
def solve(
    districts_path: str,
    sites_path: str,
    budget: int,
    radius_km: float,
    equity_weight: float,
) -> tuple[list[int], list[int]]:
    """Solves the optimization problem.

    Parameters
    ----------
    districts_path : str
        The path to the GeoJSON file containing the districts of the city.
    sites_path : str
        The path to the GeoJSON file containing the potential sites for new infrastructure.
    budget : int
        The maximum number of new infrastructure points to add.
    radius_km : float
        The radius (in kilometers) within which to consider existing infrastructure points for equity calculations.
    equity_weight : float
        The weight to assign to the equity objective in the optimization problem.

    Returns
    -------
    tuple[list[int], list[int]]
        A tuple containing two lists:
        - The first list contains the indices of the selected sites.
        - The second list contains the indices of the selected districts.
    """

    print(
        f"Solving optimization problem with districts_path={districts_path}, sites_path={sites_path}, budget={budget}, radius_km={radius_km}, equity_weight={equity_weight}..."
    )
    print("TODO")

    return ([0], [0])

In [7]:
DISTRICTS_PATH = "data/districts.geojson"
SITES_PATH = "data/sites.geojson"
BUDGET = 5
RADIUS_KM = 1.5
EQUITY_WEIGHT = 1.0
selected_sites, selected_districts = solve(
    districts_path=DISTRICTS_PATH,
    sites_path=SITES_PATH,
    budget=BUDGET,
    radius_km=RADIUS_KM,
    equity_weight=EQUITY_WEIGHT,
)

Solving optimization problem with districts_path=data/districts.geojson, sites_path=data/sites.geojson, budget=5, radius_km=1.5, equity_weight=1.0...
TODO


## 5. Visualisation des données

In [8]:
def create_map(
    districts_path: str,
    sites_path: str,
    selected_sites_indices: list[int],
    output_path: str,
) -> None:
    """Creates a map of the city with the selected locations.

    Parameters
    ----------
    districts_path : str
        The path to the GeoJSON file containing the districts of the city.
    sites_path : str
        The path to the GeoJSON file containing the potential sites for new infrastructure.
    selected_sites_indices : list[int]
        A list of indices corresponding to the selected sites in the sites GeoDataFrame.
    output_path : str
        The path where the generated map will be saved as an HTML file.
    """

    print(
        f"Creating map with parameters: districts_path={districts_path}, sites_path={sites_path}, selected_sites_indices={selected_sites_indices}, output_path={output_path}..."
    )
    print("TODO")

In [9]:
create_map(
    districts_path=DISTRICTS_PATH,
    sites_path=SITES_PATH,
    selected_sites_indices=selected_sites,
    output_path="lyon_map.html",
)

# TODO Add code to display the generated map in the notebook?

Creating map with parameters: districts_path=data/districts.geojson, sites_path=data/sites.geojson, selected_sites_indices=[0], output_path=lyon_map.html...
TODO
